In [ ]:
import os
import glob
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image

In [ ]:
# Constants:

train_path = "../Data/asl_alphabet_train/asl_alphabet_train"
test_path = "../Data/asl_alphabet_test/asl_alphabet_test/"

In [ ]:
def collect_data_paths(path: str):
        result = []
        for dir_name in os.listdir(path):
                image_path_patern = os.path.join(path, dir_name, "*.jpg")
                
                result.extend([{"image_path": ipath, "label": dir_name} for ipath in glob.glob(image_path_patern)])
        return result

In [ ]:
path_df = pd.DataFrame(collect_data_paths(train_path))
class_count = path_df['label'].value_counts().reset_index()
class_count


In [ ]:
fig = px.bar(
	class_count, 
	x='label', 
	y='count',
	title='Number of Samples per Sign Language Class',
	labels={
		'label': 'Sign / Label',
		'count': 'Image Count'
	}
 )

fig.update_layout(
    xaxis_tickangle=-45,
    template='none',
    title_font_size=20,
    bargap=0.2
)

fig.show()

As we can see there is 3000 sample for every label. Thus, this dataset is balanced.

In [ ]:
classes = sorted(class_count['label'])

rows = 10
columns = 3
IMG_WIDTH = 300
IMG_HEIGHT = 300

fig = make_subplots(
	rows=rows, 
	cols=columns, 
	subplot_titles=classes,
	horizontal_spacing=0.02,
	vertical_spacing=0.05
)

for i, c_name in enumerate(classes):
	r = (i // columns) + 1
	c = (i % columns) + 1
	
	class_df = path_df[path_df['label'] == c_name]
	random_row = class_df.sample(n=1).iloc[0]
	
	img = Image.open(random_row['image_path'])
	
	# Resize to fill the subplot
	img_resized = img.resize((IMG_WIDTH, IMG_HEIGHT))
	
	fig.add_trace(go.Image(z=img_resized), row=r, col=c)
	
	# Update axes with auto aspect ratio
	fig.update_xaxes(
		showticklabels=False, 
		showgrid=False, 
		zeroline=False, 
		scaleanchor="y",  # This keeps aspect ratio
		scaleratio=1,     # 1:1 aspect ratio
		row=r, col=c
	)
	fig.update_yaxes(
		showticklabels=False, 
		showgrid=False, 
		zeroline=False, 
		row=r, col=c
	)

fig.update_layout(
	height=rows * IMG_HEIGHT // 2,
	width=columns * IMG_WIDTH // 2,
	title_text="Random Sample Grid From Each ASL Class",
	title_font_size=20,
	showlegend=False
)

fig.show()

In [ ]:

path_df.to_csv("../Data/train_path_datafram.csv", index = False)